# Map, Filter & Reduce — Session 03: The Assembly Line

`map`, `filter`, and `reduce` are the three classical tools of functional data processing. `map` transforms every item, `filter` keeps only the items that pass a test, and `reduce` folds a whole sequence into a single value. Picture a factory line: a spray-paint station, a quality gate, and a blender at the end.

**The big picture.** Most data pipelines are a sequence of these three operations. You start with raw items, transform them into the shape you need, drop the ones that fail a rule, and finally aggregate the rest into a total, a maximum, or a report. Learning the trio gives you a vocabulary for that flow.

**Why it matters.** `map` and `filter` are lazy — they return iterators that obey the single-use rule you already met. `reduce` is eager, because collapsing many values into one cannot be lazy. Knowing when each is appropriate, and when a comprehension or a built-in is cleaner, is what separates readable Python from clever Python.

**Roadmap.** In this notebook you will:
1. Transform sequences with `map`, including two iterables at once.
2. Select items with `filter` and count matches without building a list.
3. Fold sequences with `reduce` and its `initial` argument.
4. Compare map/filter with comprehensions and pick the Pythonic option.
5. Build a real GST and inventory pipeline from a products CSV.
6. Combine map, filter, reduce, and zip on student averages.

**Key definitions.** *`map(fn, items)`*: a lazy iterator applying `fn` to every item. *`filter(pred, items)`*: a lazy iterator keeping items where `pred` is truthy. *`reduce(fn, items[, initial])`*: an eager left-to-right fold from `functools`. *Lambda*: an anonymous inline function useful for short transformations.

## 1. The Assembly Line

`map`, `filter`, and `reduce` are three machines on one production line.

| Machine | Job | Code |
|---|---|---|
| Spray-paint station | The same job on every item | `map(paint, items)` |
| Quality gate | Keeps only items that pass | `filter(ok_weight, items)` |
| Blender end | Everything in, one result out | `reduce(mix, items)` |

`map(function, iterable)` returns a lazy iterator that applies `function` to each item. It accepts more than two iterables: `map(lambda p, q: p * q, prices, qty)` pairs them like `zip` and transforms on the fly.

`filter(function, iterable)` returns a lazy iterator of the items for which `function` returns a truthy value. `reduce(function, iterable[, initial])` comes from `functools` and repeatedly combines two values until one remains — a sum, a product, a maximum, or any custom fold.

Remember the laziness rule: `map` and `filter` produce single-use iterators, so wrap them in `list()` to store or inspect them. `reduce` is eager and returns the final value directly.

In [1]:
# 🐣 MAP — spray-paint machine 🎨
prices = [100, 250, 799]
def gst(p):
    return round(p * 1.18, 2)

gsted = map(gst, prices)
print("map type   :", type(gsted).__name__, "(lazy machine)")
print("map result :", list(gsted))

# Lambda version — one-liner bina function name ke:
halves = list(map(lambda x: x / 2, prices))
print("map lambda :", halves)

# MAP with TWO iterables — zipper-transform! chain-wise superpower ⚡
qty    = [2, 1, 5]
bills  = list(map(lambda p, q: p * q, prices, qty))
print("map 2-iter :", bills, "← 100*2, 250*1, 799*5 (pei-pe-heist!)")
# Expected OUTPUT:
# map type   : map (lazy machine)
# map result : [118.0, 295.0, 942.82]
# map lambda : [50.0, 125.0, 399.5]
# map 2-iter : [200, 250, 3995] ← 100*2, 250*1, 799*5 (pei-pe-heist!)

map type   : map (lazy machine)
map result : [118.0, 295.0, 942.82]
map lambda : [50.0, 125.0, 399.5]
map 2-iter : [200, 250, 3995] ← 100*2, 250*1, 799*5 (pei-pe-heist!)


## 2. filter() — The Quality Gate

`filter(predicate, iterable)` keeps every item for which `predicate(item)` is truthy. It is the selection half of the assembly line: given scores, `filter(lambda s: s >= 50, scores)` returns only the passing ones.

The result is a lazy iterator, not a list. In a REPL it prints as a `filter` object; wrap it in `list()` to see the values. Because it is single-use, materialising it consumes it.

When you only need a count rather than the values, do not build a list at all. `sum(1 for s in scores if s >= 50)` counts matches without allocating an intermediate collection — a clean and common idiom:

```text
list(filter(pred, data))      # keep the matching values
sum(1 for x in data if pred)  # just count the matches
```

Prefer `filter` when a ready-made predicate function exists; prefer a comprehension when you would otherwise write a one-off lambda.

In [2]:
# 🐣 FILTER — quality-gate chhalni ✂️
scores = [42, 78, 55, 90, 33, 88]
passed = list(filter(lambda s: s >= 50, scores))
print("filter pass  :", passed)

names = ["apple", "kiwi", "avanade", "mango"]
a_start = list(filter(lambda w: w.startswith("a"), names))
print("filter a-    :", a_start)

# filter lazily + len via sum-1 trick (list nahi banana hota):
count_pass = sum(1 for s in scores if s >= 50)
print("genexpr count:", count_pass, "← filter-list na banao, count chahiye to ye CLASSIC! ⭐")
# Expected OUTPUT:
# filter pass  : [78, 55, 90, 88]
# filter a-    : ['apple', 'avanade']
# genexpr count: 4 ← filter-list na banao, count chahiye to ye CLASSIC! ⭐

filter pass  : [78, 55, 90, 88]
filter a-    : ['apple', 'avanade']
genexpr count: 4 ← filter-list na banao, count chahiye to ye CLASSIC! ⭐


## 3. reduce() — The Blender at the End

`reduce(function, iterable[, initial])` folds a sequence into a single value. It applies `function` to the first two items, then to that result and the third item, and so on until one value remains: `reduce(add, [1, 2, 3, 4])` computes `((1+2)+3)+4 = 10`.

`reduce` lives in `functools` and must be imported. The optional third argument is the **initial** value, placed before the first item; it also lets `reduce` handle an empty sequence, which otherwise raises `TypeError`.

| Task | `reduce` | Preferred built-in |
|---|---|---|
| Sum | `reduce(lambda a, b: a + b, xs)` | `sum(xs)` |
| Product | `reduce(lambda a, b: a * b, xs)` | `math.prod(xs)` (3.8+) |
| Maximum | `reduce(lambda a, b: a if a > b else b, xs)` | `max(xs)` |

Use `reduce` only when no built-in expresses the fold — for example, a custom accumulation over dictionaries. For sums, products, and extremes, the built-ins are clearer and faster.

In [3]:
# 🐥 REDUCE — blender-end rolling pin 🌀 (functools se import!)
from functools import reduce

nums = [1, 2, 3, 4]
total = reduce(lambda a, b: a + b, nums)
print("reduce sum   :", total, "← but! sum(nums) pythonic hai:", sum(nums))

prod = reduce(lambda a, b: a * b, nums)
print("reduce prod  :", prod, "← product ke liye math.prod() bhi hai 3.8+")

biggest = reduce(lambda a, b: a if a > b else b, nums)
print("reduce max   :", biggest, "← but max(nums):", max(nums))

# reduce with INITIAL — emergency ki saulekaudit-style:
popularity = reduce(lambda a, b: a + b, [], 0)
print("with initial :", popularity, "(empty pe bhi survives! ✅)")
# Expected OUTPUT:
# reduce sum   : 10 ← but! sum(nums) pythonic hai: 10
# reduce prod  : 24 ← product ke liye math.prod() bhi hai 3.8+
# reduce max   : 4 ← but max(nums): 4
# with initial : 0 (empty pe bhi survives! ✅)

reduce sum   : 10 ← but! sum(nums) pythonic hai: 10
reduce prod  : 24 ← product ke liye math.prod() bhi hai 3.8+
reduce max   : 4 ← but max(nums): 4
with initial : 0 (empty pe bhi survives! ✅)


### ⚖️ Map/Filter vs Comprehension — Pythonic Boxing!

| Scene | Better Choice | Why? |
|---|---|---|
| You would write `lambda x: ...` | **Comprehension** `[f(x) for x in ...]` | readable, explicit |
| A function already EXISTS (int, str.upper) | **`map(fn, ...)` clean!** — `list(map(int, s))` | no lambda needed |
| Complex condition+transform mix | **Comprehension** | both filter+map in one line |
| Lazy requirement | **genexpr / map-filter chain** — both lazy | a fresh tube |
> Guido's heart is on comprehensions 💛 — use map/filter when "the function is ready".

## 4. Real Dataset — The Products GST Pipeline

This section applies all three machines to `products_catalog.csv` (30 products). It first loads the rows into memory with `csv.DictReader`, then extracts parallel lists of names and prices.

Three operations follow:

1. **map** an 18% GST function across prices to get taxed values, then pair them with names via `zip` for display.
2. **filter** the catalogue for premium items priced above 10,000, using a comprehension that filters and pairs in one line.
3. **reduce** the entire catalogue into one inventory figure, multiplying each product's price by its stock and accumulating the total.

The example shows the natural division of labour: `map` transforms, comprehensions filter, and `reduce` aggregates. Choosing the most readable form for each step is the Pythonic goal.

In [4]:
# 💾 REAL DATASET — Products GST Pipeline! 🏭 (30 products)
import csv

def load_products():
    for path in ["../../00_Resources/datasets/products_catalog.csv",
                 "00_Python/00_Resources/datasets/products_catalog.csv"]:
        try:
            with open(path, newline="", encoding="utf-8") as f:
                return list(csv.DictReader(f))
        except FileNotFoundError:
            continue
    raise FileNotFoundError("products_catalog.csv nahi mila!")

products = load_products()
names  = [p["product_name"] for p in products]
prices = [float(p["price"]) for p in products]

# 1) GST map (18%) — first-3 print (spray-paint station!)
gsted = list(map(lambda p: round(p * 1.18, 2), prices))
print("GST first-3     :", list(zip(names, gsted))[:3])

# 2) FILTER: premium (>10,000) — quality gate!
premium = [(n, int(p)) for n, p in zip(names, prices) if p > 10_000]
print("Premium (>10k)  :", premium)

# 3) REDUCE: poora inventory value (price × stock sab ka total!)
from functools import reduce
inv_value = reduce(lambda acc, p: acc + float(p["price"]) * int(p["stock"]), products, 0)
print(f"Inventory value : ₹{inv_value:,.0f}")
# Expected OUTPUT:
# GST first-3     : [('Laptop', 56640.0), ('Smartphone', 17700.0), ('Headphones', 2360.0)]
# Premium (>10k)  : [('Laptop', 48000), ('Smartphone', 15000), ('Tablet', 22000), ('DSLR_Camera', 35000)]
# Inventory value : ₹16,629,090

FileNotFoundError: products_catalog.csv nahi mila!

## 5. Combining the Machines — Student Averages

The final example combines `map`, `filter`, `reduce`, and `zip` on `students_marks.csv`. For each student it sums five subject scores and divides by five to get an average, using `map` over the loaded rows.

- `map` builds the list of averages and the list of names.
- `zip` pairs names with averages for display.
- `filter` keeps only the distinction students, those with an average of at least 80.
- `reduce` sums the averages, which is then divided by the count to give the class average; a generator expression counts how many students scored above it.

This mirrors how real reports are assembled: transform, pair, select, aggregate. Each tool stays small, and the pipeline reads top to bottom.

In [5]:
# 💾 Students combo — map/filter/reduce/zip ALL TOGETHER! 🎓🎶
def load_students():
    for path in ["../../00_Resources/datasets/students_marks.csv",
                 "00_Python/00_Resources/datasets/students_marks.csv"]:
        try:
            with open(path, newline="", encoding="utf-8") as f:
                return list(csv.DictReader(f))
        except FileNotFoundError:
            continue
    raise FileNotFoundError("students_marks.csv nahi mila!")

students = load_students()
SUBS = ["math", "science", "english", "hindi", "computer"]

# map: har student ka AVERAGE
avgs = list(map(lambda s: sum(int(s[c]) for c in SUBS) / 5, students))
names = list(map(lambda s: s["name"], students))
print("first-3 avgs :", list(zip(names, avgs))[:3])

# filter: distinction club (avg ≥ 80) 🎖️
dist = list(filter(lambda na: na[1] >= 80, zip(names, avgs)))
print("distinction  :", dist)

# reduce: class total-avg (all avgs ka mean)
class_avg = reduce(lambda a, b: a + b, avgs) / len(avgs)
print("class avg    :", round(class_avg, 2))
print("above-avg    :", sum(1 for a in avgs if a > class_avg), "students above class-normal")
# Expected OUTPUT:
# first-3 avgs : [('Sneha_1', 57.8), ('Pooja_2', 49.8), ('Arjun_3', 69.6)]
# distinction  : [('Ananya_20', 80.2), ('Manish_29', 83.6)]
# class avg    : 65.69
# above-avg    : 27 students above class-normal

FileNotFoundError: students_marks.csv nahi mila!

## ⚠️ 5 Common Mistakes (Map/Filter/Reduce Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | `reduce` without an import — `NameError` 💥 | `from functools import reduce` FIRST |
| 2 | Indexing/slicing `filter(...)` directly | Wrap it in a list: `list(filter(...))` — a lazy machine! |
| 3 | `reduce` on an empty list (TypeError!) | `reduce(fn, [], initial)` — initial as a safety cushion |
| 4 | Consuming map/filter again (silent-empty!) — iterator habits! | wrap in list() once to store, or regenerate fresh |
| 5 | `reduce` where sum already exists | Built-in GO: `sum/max/min/math.prod` — use reduce only for custom combines |

> 🎯 **3-second verdict:** is a ready-made function there? → map. Would you write a lambda?
> → comprehension. One final single value? → builtin or reduce. ✅

## Summary

- `map(fn, items)` lazily transforms items and can take several iterables at once.
- `filter(pred, items)` lazily keeps passing items; `sum(1 for ...)` counts without a list.
- `reduce(fn, items[, initial])` eagerly folds a sequence into one value and needs an import.
- Prefer built-ins such as `sum`, `max`, and `math.prod` for standard folds, and comprehensions when you would write a one-off lambda.
- Real pipelines combine several machines: GST mapping, premium filtering, and inventory reduction over a products CSV.
- `map` and `filter` are single-use iterators; materialise them with `list()` before reusing.